# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FatimaNdeem/Flyrank-ml-internship./blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [26]:
!git clone "https://github.com/FatimaNdeem/Flyrank-ml-internship." /content/Flyrank-ml-internship.

fatal: destination path '/content/Flyrank-ml-internship.' already exists and is not an empty directory.


In [27]:
import os

print(os.listdir("/content"))

['.config', 'Flyrank-ml-internship.', 'sample_data']


In [28]:
%cd /content/Flyrank-ml-internship.

/content/Flyrank-ml-internship.


In [29]:
!ls -lh data/raw/

total 6.5M
-rw-r--r-- 1 root root 6.5M Aug 13 17:39 content_refresh_anonymized.csv


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method choice:
I chose Random Forest because this is a classification problem where the goal is to identify content that is declining. Random Forest can capture non-linear relationships between the available content and performance features without requiring strong assumptions about the relationship between them. It also provides feature importance measures that can help interpret which signals the model relies on. The model will be evaluated against the Week-4 baseline using the same data, split, and metric.

In [30]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

Dataset shape: (30000, 44)
Columns: 44


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design:
I use a grouped train/test split by client so that content from the same client does not appear in both the training and test sets. This is an honest test of whether the model can generalize to unseen clients rather than learning client-specific patterns. I use an 80/20 split and keep the same target definition and evaluation metric for the model and baseline comparison.

In [31]:
from sklearn.model_selection import GroupShuffleSplit

# Target: content is declining when recent impressions are below
# 80% of the previous 30-day impressions.
df["is_declining"] = (
    df["impressions_last_30d"]
    < 0.8 * df["impressions_prev_30d"]
).astype(int)

print("Dataset rows:", len(df))
print("Target distribution:")
print(df["is_declining"].value_counts())

# Use client_id only for grouping, not as a model feature
groups = df["client_id"]

# 80/20 grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, df["is_declining"], groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("Client overlap:", len(overlap))

Dataset rows: 30000
Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training and baseline comparison:
I trained a Random Forest classifier using the feature vector prepared in ML-05. The model is compared with the Week-4 baseline on the same test rows and using the same evaluation metric. The comparison is intended to measure whether the model provides useful decision-support beyond the rule-based baseline rather than rewarding model complexity by itself.

In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Use the feature vector created in ML-05
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

# Build features from the full dataframe using the same ML-05 approach
X = df[numeric_features + categorical_features].copy()

for col in numeric_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna("Unknown")

X = pd.get_dummies(
    X,
    columns=categorical_features,
    dtype=int
)

# Make sure train/test feature rows match the grouped split
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = df["is_declining"].iloc[train_idx]
y_test = df["is_declining"].iloc[test_idx]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

# Predictions
model_pred = model.predict(X_test)

# Model metrics
model_metrics = {
    "Accuracy": accuracy_score(y_test, model_pred),
    "Precision": precision_score(y_test, model_pred, zero_division=0),
    "Recall": recall_score(y_test, model_pred, zero_division=0),
    "F1": f1_score(y_test, model_pred, zero_division=0)
}

print("Random Forest metrics:")
for metric, value in model_metrics.items():
    print(f"{metric}: {value:.4f}")

X_train shape: (23837, 67)
X_test shape: (6163, 67)
Random Forest metrics:
Accuracy: 0.8111
Precision: 0.7929
Recall: 0.8533
F1: 0.8220


In [33]:
# Apply the Week-4 baseline logic to the same test rows

baseline_test = test_df.copy()

baseline_test["decline_score"] = (
    -baseline_test["trend_pct"]
).clip(lower=0)

baseline_test["freshness_score"] = (
    baseline_test["days_since_last_update"]
)

baseline_test["search_score"] = (
    baseline_test["search_volume"]
)

baseline_test["decline_score"] = baseline_test["decline_score"].fillna(0)
baseline_test["freshness_score"] = baseline_test["freshness_score"].fillna(0)
baseline_test["search_score"] = baseline_test["search_score"].fillna(0)

baseline_test["action_score"] = (
    0.5 * baseline_test["decline_score"]
    + 0.3 * baseline_test["freshness_score"]
    + 0.2 * baseline_test["search_score"]
)

# Same 75th-percentile rule used in ML-07,
# calculated from the training portion to avoid using test information.
threshold = (
    train_df["action_score"]
    if "action_score" in train_df.columns
    else None
)

if threshold is None:
    train_baseline = train_df.copy()

    train_baseline["decline_score"] = (
        -train_baseline["trend_pct"]
    ).clip(lower=0)

    train_baseline["freshness_score"] = (
        train_baseline["days_since_last_update"]
    ).fillna(0)

    train_baseline["search_score"] = (
        train_baseline["search_volume"]
    ).fillna(0)

    train_baseline["decline_score"] = (
        train_baseline["decline_score"].fillna(0)
    )

    train_baseline["action_score"] = (
        0.5 * train_baseline["decline_score"]
        + 0.3 * train_baseline["freshness_score"]
        + 0.2 * train_baseline["search_score"]
    )

    threshold = train_baseline["action_score"].quantile(0.75)
else:
    threshold = train_df["action_score"].quantile(0.75)

# Baseline prediction
baseline_pred = (
    baseline_test["action_score"] >= threshold
).astype(int)

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_pred,
    zero_division=0
)

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Precision": [
        baseline_precision,
        model_metrics["Precision"]
    ],
    "Recall": [
        baseline_recall,
        model_metrics["Recall"]
    ],
    "F1": [
        baseline_f1,
        model_metrics["F1"]
    ]
})

comparison

,Method,Precision,Recall,F1
0,Week-4 Baseline,0.739373,0.336932,0.462914
1,Random Forest,0.792859,0.853287,0.821964


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest performed better than the Week-4 baseline on precision, recall, and F1 on the same test split. The largest improvement was in recall, meaning the model identified substantially more of the positive cases than the baseline.

The model may still make errors when the observed signals are similar across declining and non-declining content. Some errors may also come from missing values or cases where the available content and performance signals do not fully explain the outcome. The model should therefore be treated as decision-support rather than as a final refresh decision.

The most important features will be reviewed using feature importance to understand which observed signals the model relies on.

In [34]:
# Feature importance from the Random Forest

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

print("Top 10 features:")
feature_importance.head(10)

Top 10 features:


,feature,importance
18,impressions_prev_30d,0.163577
15,impressions_last_30d,0.130264
5,impressions_90d,0.064291
24,avg_position,0.054624
13,days_with_impressions,0.049839
21,content_age_days,0.041282
17,sessions_last_30d,0.028462
23,ctr,0.025397
16,clicks_last_30d,0.025231
4,char_count,0.024692


In [35]:
# Error analysis on the test set

error_analysis = test_df[
    ["content_id", "is_declining"]
].copy()

error_analysis["prediction"] = model_pred

error_analysis["error_type"] = "Correct"

error_analysis.loc[
    (error_analysis["is_declining"] == 1) &
    (error_analysis["prediction"] == 0),
    "error_type"
] = "False Negative"

error_analysis.loc[
    (error_analysis["is_declining"] == 0) &
    (error_analysis["prediction"] == 1),
    "error_type"
] = "False Positive"

print("Error counts:")
print(error_analysis["error_type"].value_counts())

print("\nFalse negatives:")
print(
    error_analysis[
        error_analysis["error_type"] == "False Negative"
    ].head(5)
)

print("\nFalse positives:")
print(
    error_analysis[
        error_analysis["error_type"] == "False Positive"
    ].head(5)
)

Error counts:
error_type
Correct           4999
False Positive     702
False Negative     462
Name: count, dtype: int64

False negatives:
               content_id  is_declining  prediction      error_type
81   content_16788821b64a             1           0  False Negative
129  content_b4170c25efd2             1           0  False Negative
148  content_033581b09704             1           0  False Negative
165  content_eaea09d6891e             1           0  False Negative
252  content_aba4b4460e47             1           0  False Negative

False positives:
               content_id  is_declining  prediction      error_type
13   content_a5a2fbc76336             0           1  False Positive
26   content_72c5c2d73e5a             0           1  False Positive
36   content_bce275871a25             0           1  False Positive
204  content_976d5deeab73             0           1  False Positive
277  content_714b0fd9ee80             0           1  False Positive


The model made 702 false-positive predictions and 462 false-negative predictions on the test set. False positives are cases predicted as declining when the observed target was not declining, while false negatives are declining cases that the model did not identify. The model therefore missed fewer positive cases than the number of false positives it produced. Feature importance showed that impressions_prev_30d and impressions_last_30d were the strongest observed signals, followed by impressions_90d and avg_position. These results are directional and should be treated as decision-support rather than proof that content must be refreshed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.